# §35: zero-shot read of K5f, K5dfi, K5dfie **and the released LaGoVAD checkpoint** on **whole DoTA-CAP** (dev 569 + eval 560 = 1,129 clips)

Rule: `core/docs/final/KIM_RUN.md` **§35**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **PRELIMINARY (n = 1 seed, s2024); printed, decides nothing; K5f stays the reference.**

**Opens a sealed split.** `dota_cap_eval` is in `V2_SEALED_SPLITS`; this notebook reads it with `--final`, **by the user's request (2026-11-21)**. v2's Final step already opened it once for A0-A3 (`OPENED.json`, 2026-10-06); this is the first read of a KIM arm, and of the released LaGoVAD checkpoint, on it. No arm is chosen from this read.

**No extraction is needed if v2 §29 ran.** `nexar_zeroshot.ipynb` (v2 §29) already baked `A2_S/dota_cap_eval_s3_from_s1_nexar` (CLIP `DoTA_s1_ncc` + V2-S `DoTA_CAP_s1_squash`, Nexar statistics, `s1[::3]`) and read it with A2 (`outputs/v2/v2_nexar_window/eval/dota/A2/dota_cap_eval`, 560 clips). That bake is K5's exact input. Step 3 checks it; only if it is missing does step 4 rebuild it on CPU from the two feature caches. If the **V2-S features of the eval clips** are missing, the notebook stops: producing them streams the CAP tars (`colab/v2/dota_cap_videomae.ipynb`), and that is not improvised here. **LaGoVAD needs no bake**: it reads the plain CLIP rows `DoTA_s1_ncc[::3]`, as in its dev read (`zs_lagovad.ipynb` step 6, protocol B).

| row | what | checkpoint (read only) | dev read (569) |
|---|---|---|---|
| **K5f** | reference (§31.1) | `kim_fixed/K5f/K5/s2024` | `kim_fixed/K5f/eval/dota/K5` |
| **K5dfi** | K5df + `loss.dvs_anchor_mode=ignore` (§34) | `kim_dvs_ignore/K5dfi/K5/s2024` | `kim_dvs_ignore/K5dfi/eval/dota/K5` |
| **K5dfie** | K5dfi + `data.is_egocentric=true` (§34) | `kim_dvs_ignore/K5dfie/K5/s2024` | `kim_dvs_ignore/K5dfie/eval/dota/K5` |
| **LaGoVAD** | released checkpoint, the comparison baseline (`ZS_TAD_DADA.md` §7) | `Thesis/ckpts/best.ckpt` | `zs_lagovad/eval/dota/LaGoVAD/B` |

| step | what | GPU | estimate |
|---|---|---|---|
| 1-2 | setup, deps | no | 2 min |
| 3 | preflight: J10 + config of the 3 KIM checkpoints, LaGoVAD's checkpoint + dev read, the eval bake, features, dev reads | no | 1-2 min |
| 4 | **only if the bake is missing:** rebuild it (CPU) | no | 5-10 min |
| 5 | `OPENED.json` marker, then 3 x `protocol_b_eval --final` (KIM rows, bake) + 1 x `--baseline --final` (LaGoVAD, plain CLIP; its rows are first checked equal to the bake's CLIP columns) | yes | 4 x 3-6 min |
| 6 | whole DoTA-CAP = the dev read (569) + this read (560), per clip, no re-scoring | no | < 1 min |
| 7 | `zs_arm_table` on whole DoTA-CAP only (control K5f; LaGoVAD a row) -> **paste `REPORTS/zs_arm_table_cap_all_s2024.md` to Claude** | no | 2-5 min |

Everything new goes to `Thesis-Final/outputs/kim_cap_eval/`. A re-run skips what is on Drive (a split is opened once). **Watch the log; on any failure paste the printed tail.**

**Reading rules (written now).** Macro = mean per-clip AUC (frame level, two-class clips), 95 % source-video cluster CI. Micro / AP after per-clip min-max (every DoTA-CAP clip holds an accident). Name every number **DoTA-CAP (n/1397)** (D14), never plain "DoTA", and never beside LaGoVAD's printed 62.60: the LaGoVAD row here is the released checkpoint re-read under protocol B (lesson 8b). **One table only: whole DoTA-CAP (user, 2026-11-21).** The eval read of step 5 is the missing half, not a reported set: no eval-only number is printed or tabled.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, DoTA_s1_ncc, ckpts/best.ckpt (LaGoVAD released)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 bakes (v2 §29), V2-S caches
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, the three checkpoints and every dev read; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert 'released LaGoVAD' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §35 with the LaGoVAD row -- re-upload kat-vad/'
assert '--final' in (REPO / 'core/tools/protocol_b_eval.py').read_text(), 'protocol_b_eval has no --final -- re-upload kat-vad/'
assert (REPO / 'core/tools/zs_arm_table.py').is_file(), 'no zs_arm_table -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                       # "K5": names the eval sub-folders only (eval/<set>/K5)
K5F      = constants.KIM_REFERENCE_ARM                  # the reference and the control of every contrast here
K5DFI, K5DFIE = constants.KIM_DVS_IGNORE_ROW, constants.KIM_DVS_IGNORE_EGO_ROW
ROWS     = (K5F, K5DFI, K5DFIE)                         # the KIM rows (checkpoints with metrics.jsonl, the bake)
LAG, LAG_RUN = constants.ZS_LAGOVAD_ARM, constants.ZS_LAGOVAD_RUN   # 'LaGoVAD', 'released'
ALL_ROWS = (*ROWS, LAG)                                 # every row of the tables
SEED     = constants.KIM_SEED                           # 2024
ENC      = constants.V5_ENCODER                         # V2-S
CAP_DEV, CAP_EVAL = constants.V2_SPLIT_DOTA_CAP_DEV, constants.V2_SPLIT_DOTA_CAP_EVAL
STRIDE   = constants.NEXAR_EVAL_STRIDE                  # protocol B: s1[::3]
assert STRIDE == constants.V2_E1_STRIDE_BC, (STRIDE, constants.V2_E1_STRIDE_BC)   # LaGoVAD's dev read used the latter
N_DEV, N_EVAL = 569, 560                                # the frozen DOTA_CAP_MANIFEST counts

ROW_ROOT = {K5F: constants.OUTPUT_ROOT / constants.KIM_FIXED_OUTPUT_DIRNAME / K5F,
            K5DFI: constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFI,
            K5DFIE: constants.OUTPUT_ROOT / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFIE}
LAG_CKPT = Path(DRIVE) / 'ckpts' / 'best.ckpt'          # the file of LaGoVAD's dev read (and of Gate T0)
LAG_PROTO = 'B'                                         # zs_lagovad.ipynb wrote eval/dota/LaGoVAD/{B,A}; B = s1[::3]


def run_dir(row):
    return ROW_ROOT[row] / K5 / f's{SEED}'


def dev_dir(row):
    '''The row's own earlier read of DoTA-CAP-dev (569 clips, protocol B).'''
    if row == LAG:
        return constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME / 'eval' / 'dota' / LAG / LAG_PROTO
    return ROW_ROOT[row] / 'eval' / 'dota' / K5


V2_CACHE = Path(DRIVE_V2) / 'cache'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                       # the Nexar statistics K5 was trained with
BAKE_NAME = f'{CAP_EVAL}_s{STRIDE}_from_s1_nexar'
BAKE_V2  = V2_CACHE / 'v2' / 'A2_S' / BAKE_NAME                        # written by v2 §29 (nexar_zeroshot.ipynb)
BAKE_F   = Path(DRIVE_F) / 'cache' / 'v2' / 'A2_S' / BAKE_NAME         # this notebook's fallback
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
CAP_S1_NAME = f'{constants.DOTA_CAP_DATASET}_s1_squash'
CAP_S1_CANDIDATES = [Path(root) / 'cache' / 'video' / ENC / CAP_S1_NAME for root in (DRIVE, DRIVE_V2, DRIVE_F)]

OUT      = constants.OUTPUT_ROOT / 'kim_cap_eval'
REPORTS  = OUT / 'REPORTS'
LOCAL    = Path('/content/k5_cap_eval')
LOCAL.mkdir(parents=True, exist_ok=True)
print('code', REPO, '| outputs', OUT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 5 needs one'
df -h /content | tail -1

In [ ]:
import numpy as np
import torch
import yaml

from core.data.v2_splits import load_split           # noqa: E402
from core.tools.v5_stage1 import _flat                # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


def missing_ids(d, ids):
    return [v for v in ids if not (d / f'{v}.npy').exists()]

## 3. Preflight (CPU): the checkpoints, the eval bake, the features, the dev reads — small files only

In [ ]:
# ids only: the sealed split is named here on purpose (user, 2026-11-21); no score is read in this cell
EVAL_IDS = sorted(load_split(CAP_EVAL, final=True))
DEV_IDS = sorted(load_split(CAP_DEV))
assert (len(DEV_IDS), len(EVAL_IDS)) == (N_DEV, N_EVAL), (len(DEV_IDS), len(EVAL_IDS))
assert not set(DEV_IDS) & set(EVAL_IDS), 'dev and eval overlap'
DEV_READ_FILES = ('clip_aucs.json', 'clip_scores.npz', 'protocol_b_readout.json')

bad = []
STEPS = None
for row in ROWS:
    d = run_dir(row)
    step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
    ver = json.loads((d / 'verified.json').read_text()) if (d / 'verified.json').exists() else None
    if not (last and ver and step == last['global_step'] == ver['step']):
        bad.append(f'{row}: J10 (ckpt {step}, metrics {last and last["global_step"]}, verified {ver and ver["step"]})')
        continue
    STEPS = STEPS or step
    if step != STEPS:
        bad.append(f'{row}: step {step} != {STEPS}')
    cfg = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    want = {'kim.enabled': True, 'kim.prototypes': True, 'kim.edge_mask': True, 'train.seed': SEED,
            'v2.crn': constants.V2_OFF, 'v2.motion': ENC}
    if row == K5F:
        want |= {'dvs.theta': 1.0}
    else:
        want |= {'dvs.theta': 0.7, 'dvs.theta_ego': 0.85, 'kim.seam_mask': True,
                 'loss.dvs_anchor_mode': constants.DVS_ANCHOR_MODE_IGNORE,
                 'data.is_egocentric': row == K5DFIE}
    wrong = {k: (cfg.get(k), v) for k, v in want.items() if cfg.get(k) != v}
    if wrong:
        bad.append(f'{row}: config (got, want) {wrong}')
    bad += [f'{row}: no dev read {dev_dir(row) / f}' for f in DEV_READ_FILES if not (dev_dir(row) / f).exists()]

# LaGoVAD: the released file its dev read scored, and that dev read (protocol B, s1[::3], 569 clips)
if not LAG_CKPT.is_file():
    bad.append(f'{LAG}: no released checkpoint at {LAG_CKPT}')
bad += [f'{LAG}: no dev read {dev_dir(LAG) / f}' for f in DEV_READ_FILES if not (dev_dir(LAG) / f).exists()]
if (dev_dir(LAG) / 'protocol_b_readout.json').exists():
    lag_dev = json.loads((dev_dir(LAG) / 'protocol_b_readout.json').read_text())
    got = (lag_dev['split'], lag_dev['clips'], lag_dev['released_baseline'], lag_dev['runs'],
           lag_dev['checkpoints'], f's1[::{STRIDE}]' in lag_dev['protocol'])
    want_lag = (CAP_DEV, N_DEV, True, [LAG_RUN], {LAG_RUN: str(LAG_CKPT)}, True)
    if got != want_lag:
        bad.append(f'{LAG}: dev read is not the released ckpt at protocol B: got {got}, want {want_lag}')
assert not bad, 'preflight failed -- paste to Claude:\n' + '\n'.join(bad)
print(f'3 KIM checkpoints OK at step {STEPS}; LaGoVAD {LAG_CKPT.name} ({LAG_CKPT.stat().st_size / 2**20:.1f} MiB); 4 dev reads present')

# the eval input: CLIP rows for native frame counts, and the bake K5 reads (A2, Nexar statistics)
SHA = manifest(A2_NEXAR)['train_ids_sha1']
assert (manifest(A2_NEXAR)['crn'], manifest(A2_NEXAR)['motion']) == (constants.V2_OFF, ENC), manifest(A2_NEXAR)
miss = missing_ids(DOTA_S1, EVAL_IDS)
assert not miss, f'{len(miss)} DoTA-CAP-eval clips have no CLIP s1 rows: {miss[:5]}'


def bake_ok(d):
    if not (d / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        return False
    m = manifest(d)
    return ((m['crn'], m['motion'], m['train_ids_sha1'], m['baked_ids']) == (constants.V2_OFF, ENC, SHA, N_EVAL)
            and not missing_ids(d, EVAL_IDS))


BAKE = next((d for d in (BAKE_V2, BAKE_F) if bake_ok(d)), None)
CAP_S1 = next((d for d in CAP_S1_CANDIDATES if d.is_dir() and not missing_ids(d, EVAL_IDS)), None)
if BAKE:
    print('eval bake found:', BAKE, '-> no extraction, no baking')
elif CAP_S1:
    print('eval bake missing; V2-S features of every eval clip found at', CAP_S1, '-> step 4 rebuilds the bake (CPU)')
else:
    raise AssertionError(
        f'No eval bake ({BAKE_V2}) and no V2-S features of the {N_EVAL} eval clips under {[str(p) for p in CAP_S1_CANDIDATES]}. '
        'They come from `colab/v2/dota_cap_videomae.ipynb` (streams the CAP tars; hours). Paste this to Claude, do not improvise.')

## 4. Only if the eval bake is missing: rebuild it with Nexar's statistics (CPU, → Drive)

`build_v2_inputs apply` exactly as v2 §29: Nexar-fitted `s`, `c`, CRN off, V2-S motion, `s1[::3]` of the DoTA CLIP rows. Skipped when step 3 found the bake.

In [ ]:
if BAKE is None:
    ids_file = LOCAL / 'ids' / f'{CAP_EVAL}.txt'
    ids_file.parent.mkdir(parents=True, exist_ok=True)
    ids_file.write_text(''.join(f'{v}\n' for v in EVAL_IDS))
    local = LOCAL / 'bake'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply',
         '--stats-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'stats'),
         '--clip-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', EVAL_IDS),
         '--video-dir', stage(CAP_S1, LOCAL / 'stage' / 'cap_v2s', EVAL_IDS),
         '--ids-file', ids_file, '--stride', STRIDE, '--out-dir', local], log=LOCAL / 'bake.log')
    publish(local, BAKE_F)
    assert bake_ok(BAKE_F), f'rebuilt bake fails its own check: {manifest(BAKE_F)}'
    BAKE = BAKE_F
    print('bake rebuilt ->', BAKE)
else:
    print('bake already present:', BAKE)

## 5. Open `dota_cap_eval` (marker), then read K5f, K5dfi, K5dfie and LaGoVAD with `protocol_b_eval --final`

KIM rows: same call as the dev read (`k5_dvs_ignore.ipynb` step 7) with `--split dota_cap_eval --final`; the checkpoint's `v2.crn` / `v2.motion` must match the bake's manifest (the tool refuses otherwise).

LaGoVAD: same call as its dev read (`zs_lagovad.ipynb` step 6, protocol B) with `--split dota_cap_eval --final`: `--baseline released best.ckpt` on the plain CLIP rows `DoTA_s1_ncc` at `--stride 3`. Before it is scored, `zs_lagovad_readout check-inputs` asserts that those plain rows equal the bake's first 512 columns on every eval clip (§7.3-1): every row sees the same CLIP frames, so the contrast is the model, not the input.

In [ ]:
MARK = OUT / 'OPENED.json'
CKPT_OF = {**{r: run_dir(r) / 'checkpoint_last.pt' for r in ROWS}, LAG: LAG_CKPT}
if MARK.exists():
    print('dota_cap_eval already opened for this line:', MARK.read_text())
else:
    OUT.mkdir(parents=True, exist_ok=True)
    MARK.write_text(json.dumps({
        'opened_utc': time.strftime('%Y-%m-%dT%H:%MZ', time.gmtime()), 'split': CAP_EVAL, 'rows': list(ALL_ROWS), 'seed': SEED,
        'authorized': 'user, 2026-11-21', 'rule': 'KIM_RUN.md §35: printed, decides nothing',
        'checkpoints': {r: {'path': str(p), 'bytes': p.stat().st_size} for r, p in CKPT_OF.items()}}, indent=2))
    print('opened:', MARK)

L_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1_eval', EVAL_IDS)
L_IN = stage(BAKE, LOCAL / 'stage' / 'cap_eval_bake', EVAL_IDS)
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
IDS_FILE = LOCAL / 'ids' / f'{CAP_EVAL}.txt'
IDS_FILE.parent.mkdir(parents=True, exist_ok=True)
IDS_FILE.write_text(''.join(f'{v}\n' for v in EVAL_IDS))


def eval_dir(row, name=CAP_EVAL):
    return OUT / row / 'eval' / name / (LAG_PROTO if row == LAG else K5)


def score_args(row):
    '''protocol_b_eval arguments that differ by row: the checkpoint and the input rows.'''
    if row == LAG:      # released checkpoint, plain CLIP rows, protocol B (as zs_lagovad.ipynb step 6)
        return ['--baseline', LAG_RUN, LAG_CKPT, '--input-dir', L_S1, '--stride', STRIDE]
    return ['--run', f's{SEED}', run_dir(row) / 'checkpoint_last.pt', '--input-dir', L_IN]


def check_eval_read(row, r):
    assert r['split'] == CAP_EVAL and r['clips'] == N_EVAL, (r['split'], r['clips'])
    assert f's1[::{STRIDE}]' in r['protocol'], r['protocol']
    if row == LAG:
        assert r['released_baseline'] and r['runs'] == [LAG_RUN], (r['released_baseline'], r['runs'])
        assert r['checkpoints'] == {LAG_RUN: str(LAG_CKPT)}, r['checkpoints']
    else:
        assert r['global_steps'] == {f's{SEED}': STEPS}, r['global_steps']
        assert (r['arm']['crn'], r['arm']['motion']) == (constants.V2_OFF, ENC), r['arm']


# LaGoVAD's plain CLIP rows must be the KIM rows' CLIP columns on every eval clip (§7.3-1)
run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L_S1,
     '--bake', L_IN, '--ids-file', IDS_FILE, '--plain-stride', STRIDE])

for row in ALL_ROWS:
    out = eval_dir(row)
    if not (out / 'protocol_b_readout.json').exists():
        local = LOCAL / 'eval' / row
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', *score_args(row),
             '--final', '--s1-dir', L_S1, '--metadata', DOTA / 'metadata_val.json',
             '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', CAP_EVAL,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_{CAP_EVAL}.log')
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    check_eval_read(row, r)
    print(f"{row}: eval half read ({r['clips']} clips, {r['two_class']} two-class); numbers only on whole DoTA-CAP (step 7)")
flush_drive()

## 6. Whole DoTA-CAP = the row's dev read (569, its own earlier run) + the eval read (560)

No re-scoring: per-clip curves and per-clip AUCs are concatenated. **Guards:** same checkpoint and step (KIM rows) or the same released file (LaGoVAD), same arm, disjoint ids, 569 + 560 clips. Each row's dev read used the same input recipe as its eval read (KIM: the Nexar-statistics bake at `s1[::3]`; LaGoVAD: plain `DoTA_s1_ncc[::3]`), so the two halves are one protocol.

In [ ]:
from core.tools import protocol_b_eval as pb  # noqa: E402


def check_same_model(row, rd, re_):
    '''The dev read and the eval read scored the same checkpoint with the same arm.'''
    assert rd['arm'] == re_['arm'], (rd['arm'], re_['arm'])
    assert rd['global_steps'] == re_['global_steps'], (rd['global_steps'], re_['global_steps'])
    assert rd['runs'] == re_['runs'], (rd['runs'], re_['runs'])
    if row == LAG:
        assert rd['released_baseline'] and re_['released_baseline'], 'LaGoVAD: a read is not the released baseline'
        assert rd['checkpoints'] == re_['checkpoints'] == {LAG_RUN: str(LAG_CKPT)}, (rd['checkpoints'], re_['checkpoints'])
        return
    assert rd['global_steps'] == {f's{SEED}': STEPS}, rd['global_steps']
    for r in (rd, re_):
        ck = Path(r['checkpoints'][f's{SEED}']).parts
        assert ck[-4:] == (row, K5, f's{SEED}', 'checkpoint_last.pt'), f'{row}: read a different checkpoint {ck[-4:]}'


def pooled(row):
    dev, ev, out = dev_dir(row), eval_dir(row), eval_dir(row, 'dota_cap_all')
    if (out / pb.READOUT_JSON).exists():
        return out
    rd, re_ = (json.loads((d / pb.READOUT_JSON).read_text()) for d in (dev, ev))
    check_same_model(row, rd, re_)
    sd, ld = pb.read_clip_scores(dev / pb.CLIP_SCORES_NPZ)
    se, le = pb.read_clip_scores(ev / pb.CLIP_SCORES_NPZ)
    assert (len(sd), len(se)) == (N_DEV, N_EVAL) and not set(sd) & set(se), (len(sd), len(se))
    aucs = {**json.loads((dev / pb.CLIP_AUCS_JSON).read_text()), **json.loads((ev / pb.CLIP_AUCS_JSON).read_text())}
    local = LOCAL / 'pooled' / row
    shutil.rmtree(local, ignore_errors=True)
    local.mkdir(parents=True)
    pb.write_clip_scores(local / pb.CLIP_SCORES_NPZ, {**sd, **se}, {**ld, **le})
    (local / pb.CLIP_AUCS_JSON).write_text(json.dumps(aucs, indent=2))
    (local / pb.READOUT_JSON).write_text(json.dumps({
        'protocol': rd['protocol'], 'split': f'{CAP_DEV} + {CAP_EVAL}', 'clips': N_DEV + N_EVAL, 'two_class': len(aucs),
        'arm': rd['arm'], 'runs': rd['runs'], 'released_baseline': rd.get('released_baseline', False),
        'global_steps': rd['global_steps'], 'checkpoints': {'dev': rd['checkpoints'], 'eval': re_['checkpoints']},
        'per_run_macro': {rd['runs'][0]: float(np.mean(list(aucs.values())))}, 'position_ruler': None,
        'note': 'concatenation of the dev read and the eval read; no re-scoring'}, indent=2))
    publish(local, out)
    return out


for row in ALL_ROWS:
    p = pooled(row)
    r = json.loads((p / pb.READOUT_JSON).read_text())
    print(f"{row} whole DoTA-CAP: {r['clips']} clips, {r['two_class']} two-class, "
          f"mean per-clip AUC {next(iter(r['per_run_macro'].values())):.4f}")

## 7. `zs_arm_table`: macro [CI], micro, AP and paired deltas vs K5f — on whole DoTA-CAP (1,129)

Rows K5f (control), K5dfi, K5dfie, **LaGoVAD (released checkpoint)**. `LaGoVAD - K5f` is printed as the table's Δ (arm − control); read it with its sign flipped for "K5f over LaGoVAD". The tool refuses rows with different clips or different frame labels, so all four are paired on the same clips. The tool's labels for this set are static ("DoTA-CAP-dev", "569 clips"); they are set per run below, and the written report is relabeled. Printed only; it decides nothing. **Paste `REPORTS/zs_arm_table_cap_all_s2024.md` to Claude.**

In [ ]:
from core.tools import zs_arm_table as zat  # noqa: E402

PROTOCOL_TXT = 'protocol B; micro / AP after per-clip min-max'


def table(name, subdir, label, n):
    local = LOCAL / 'report'
    old_pool, old_name = zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA]
    zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA] = f'{n} clips, {PROTOCOL_TXT}', label
    try:
        reads = [a for row in ALL_ROWS for a in ('--read', zat.DOTA, row, str(eval_dir(row, subdir)))]
        zat.main([*reads, '--control', K5F, '--seed', str(SEED), '--name', name, '--out-dir', str(local)])
    finally:
        zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA] = old_pool, old_name
    md_path = local / f'{constants.ZS_ARM_TABLE_STEM}_{name}.md'
    md_path.write_text(md_path.read_text().replace('DoTA-CAP-dev and DADA-2000 after', f'{label} after'))
    return json.loads((local / f'{constants.ZS_ARM_TABLE_STEM}_{name}.json').read_text())['sets'][zat.DOTA]


T_ALL = table(f'cap_all_s{SEED}', 'dota_cap_all', 'DoTA-CAP (dev+eval)', N_DEV + N_EVAL)
publish(LOCAL / 'report', REPORTS)

for title, t in (('whole DoTA-CAP, dev + eval (1,129)', T_ALL),):
    print(f"\n{title} -- control {K5F}, s{SEED}, n = 1, printed only")
    print(f"{'row':8s} {'macro [95 % CI]':26s} {'micro':>7s} {'AP':>7s}   (base rate {next(iter(t['cells'].values()))['base_rate']:.4f})")
    for row, c in t['cells'].items():
        m = c['macro']
        print(f"{row:8s} {m['mean']:.4f} [{m['low']:.4f}, {m['high']:.4f}]   {c['micro']:.4f}  {c['ap']:.4f}")
    for row, d in t['deltas'].items():
        print(f"  {row} - {K5F}: macro {d['macro']['mean']:+.4f} ({d['macro_sentence']}) | micro {d['micro']['mean']:+.4f} ({d['micro_sentence']})")

print('\n' + (REPORTS / f'{constants.ZS_ARM_TABLE_STEM}_cap_all_s{SEED}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste {REPORTS}/zs_arm_table_cap_all_s{SEED}.md to Claude')